# Option C : LSTM avec Données Externes

**Approche enrichie** : Combine les données OHLCV avec des données externes pour capturer le sentiment du marché.

**Données externes incluses :**
- Fear & Greed Index (gratuit via API)
- Google Trends (recherches "Bitcoin")
- Données de marché étendues (S&P500, Gold, DXY pour corrélations)

**Avantages :**
- Capture le sentiment du marché
- Corrélations inter-marchés
- Meilleure contextualisation

**Runtime recommandé:** GPU T4

In [ ]:
!pip install yfinance pandas numpy scikit-learn tensorflow matplotlib requests pytrends --quiet
print("Dependencies installed!")

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import json
import joblib
import requests
import warnings
import time
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras
from keras.models import Model
from keras.layers import LSTM, Dense, Dropout, BatchNormalization, Input, Concatenate, Bidirectional, Attention, MultiHeadAttention, LayerNormalization, GRU
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from keras.optimizers import Adam

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ============ CONFIGURATION ============
CONFIG = {
    "symbol": "BTC-USD",
    "years_of_data": 3,
    "sequence_length": 21,           # 3 semaines
    
    # Actifs corrélés
    "correlated_assets": [
        "^GSPC",    # S&P 500
        "GC=F",     # Gold
        "DX-Y.NYB", # Dollar Index
        "^VIX",     # Volatility Index
        "ETH-USD",  # Ethereum (très corrélé)
    ],
    
    "target_columns": ["Open", "High", "Low", "Close"],
    
    # Modèle
    "lstm_units": 128,
    "dropout_rate": 0.3,
    
    # Entraînement
    "batch_size": 32,
    "epochs": 150,
    "learning_rate": 0.001,
    "validation_split": 0.15,
    "test_split": 0.1,
}

print("Configuration Option C (External Data):")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Télécharger les données BTC
end_date = datetime.now()
start_date = end_date - timedelta(days=CONFIG["years_of_data"] * 365)

print(f"Downloading {CONFIG['symbol']} data...")
df_btc = yf.download(CONFIG["symbol"], start=start_date, end=end_date, progress=False)
df_btc = df_btc.dropna()
print(f"  BTC: {len(df_btc)} days")

In [ ]:
# Télécharger les actifs corrélés
correlated_data = {}

for symbol in CONFIG["correlated_assets"]:
    print(f"Downloading {symbol}...")
    try:
        df_temp = yf.download(symbol, start=start_date, end=end_date, progress=False)
        if len(df_temp) > 100:
            correlated_data[symbol] = df_temp['Close']
            print(f"  {symbol}: {len(df_temp)} days")
        else:
            print(f"  {symbol}: Not enough data, skipping")
    except Exception as e:
        print(f"  {symbol}: Failed - {e}")
    time.sleep(0.5)  # Rate limiting

print(f"\nSuccessfully loaded {len(correlated_data)} correlated assets")

In [ ]:
# Télécharger Fear & Greed Index
print("\nDownloading Fear & Greed Index...")

try:
    # API Alternative.me (gratuite)
    url = "https://api.alternative.me/fng/?limit=0&format=json"
    response = requests.get(url, timeout=30)
    fng_data = response.json()['data']
    
    # Convertir en DataFrame
    fng_df = pd.DataFrame(fng_data)
    fng_df['timestamp'] = pd.to_datetime(fng_df['timestamp'].astype(int), unit='s')
    fng_df = fng_df.set_index('timestamp')
    fng_df['value'] = fng_df['value'].astype(float)
    fng_df = fng_df.sort_index()
    
    print(f"  Fear & Greed: {len(fng_df)} days")
    print(f"  Range: {fng_df['value'].min():.0f} - {fng_df['value'].max():.0f}")
except Exception as e:
    print(f"  Fear & Greed: Failed - {e}")
    fng_df = None

In [ ]:
# Fusionner toutes les données
df = df_btc.copy()

# Ajouter les actifs corrélés
for symbol, series in correlated_data.items():
    col_name = symbol.replace('^', '').replace('=F', '').replace('-', '_').replace('.', '_')
    df[col_name] = series

# Ajouter Fear & Greed Index
if fng_df is not None:
    df['FearGreed'] = fng_df['value']

# Forward fill pour les jours manquants (weekends, etc.)
df = df.ffill()

# Supprimer les lignes avec trop de NaN
df = df.dropna(thresh=len(df.columns) - 2)
df = df.fillna(method='ffill').fillna(method='bfill')

print(f"\nMerged dataset: {len(df)} days, {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")

In [ ]:
# Feature Engineering
df_features = df.copy()

# BTC Features
df_features['BTC_Returns'] = df_features['Close'].pct_change()
df_features['BTC_Returns_5d'] = df_features['Close'].pct_change(5)
df_features['BTC_Volatility'] = df_features['BTC_Returns'].rolling(window=7).std()
df_features['BTC_Range'] = (df_features['High'] - df_features['Low']) / df_features['Close']
df_features['BTC_Volume_Ratio'] = df_features['Volume'] / df_features['Volume'].rolling(window=20).mean()

# RSI simplifié
delta = df_features['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / (loss + 1e-8)
df_features['BTC_RSI'] = 100 - (100 / (1 + rs))

# Moyennes mobiles ratio
df_features['BTC_SMA_7'] = df_features['Close'].rolling(window=7).mean()
df_features['BTC_SMA_21'] = df_features['Close'].rolling(window=21).mean()
df_features['BTC_MA_Ratio'] = df_features['BTC_SMA_7'] / df_features['BTC_SMA_21']

# Corrélations features (returns des actifs corrélés)
for col in df_features.columns:
    if col in ['GSPC', 'GC_F', 'DX_Y_NYB', 'VIX', 'ETH_USD']:
        df_features[f'{col}_Returns'] = df_features[col].pct_change()

# Fear & Greed features
if 'FearGreed' in df_features.columns:
    df_features['FearGreed_Change'] = df_features['FearGreed'].diff()
    df_features['FearGreed_MA'] = df_features['FearGreed'].rolling(window=7).mean()

# Supprimer les NaN
df_features = df_features.dropna()

print(f"\nFeatures dataset: {len(df_features)} days, {len(df_features.columns)} columns")

In [ ]:
# Visualiser les corrélations
import seaborn as sns

# Sélectionner les colonnes numériques clés
corr_cols = ['Close', 'BTC_Returns', 'BTC_RSI', 'BTC_Volatility']
if 'GSPC' in df_features.columns:
    corr_cols.append('GSPC')
if 'ETH_USD' in df_features.columns:
    corr_cols.append('ETH_USD')
if 'FearGreed' in df_features.columns:
    corr_cols.append('FearGreed')
if 'VIX' in df_features.columns:
    corr_cols.append('VIX')

corr_matrix = df_features[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdYlGn', center=0, fmt='.2f')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Sélectionner les features finales
# Commencer avec les features de base et ajouter les externes disponibles
feature_cols = ['Open', 'High', 'Low', 'Close', 'Volume',
                'BTC_Returns', 'BTC_Returns_5d', 'BTC_Volatility', 
                'BTC_Range', 'BTC_Volume_Ratio', 'BTC_RSI', 'BTC_MA_Ratio']

# Ajouter les features externes si disponibles
for col in ['GSPC', 'ETH_USD', 'VIX', 'GC_F', 'DX_Y_NYB', 
            'GSPC_Returns', 'ETH_USD_Returns', 'FearGreed', 'FearGreed_Change', 'FearGreed_MA']:
    if col in df_features.columns:
        feature_cols.append(col)

print(f"Selected {len(feature_cols)} features:")
for i, col in enumerate(feature_cols):
    print(f"  {i+1}. {col}")

In [ ]:
# Préparer les données
target_cols = CONFIG["target_columns"]
seq_len = CONFIG["sequence_length"]

# Extraire les données
data = df_features[feature_cols].values
targets = df_features[target_cols].values

# Utiliser RobustScaler (moins sensible aux outliers)
scaler_features = RobustScaler()
scaler_targets = MinMaxScaler(feature_range=(0, 1))

data_scaled = scaler_features.fit_transform(data)
targets_scaled = scaler_targets.fit_transform(targets)

# Créer les séquences
X, y = [], []
for i in range(seq_len, len(data_scaled)):
    X.append(data_scaled[i-seq_len:i])
    y.append(targets_scaled[i])

X = np.array(X)
y = np.array(y)

# Split temporel
test_size = int(len(X) * CONFIG["test_split"])
val_size = int(len(X) * CONFIG["validation_split"])
train_size = len(X) - test_size - val_size

X_train, y_train = X[:train_size], y[:train_size]
X_val, y_val = X[train_size:train_size+val_size], y[train_size:train_size+val_size]
X_test, y_test = X[train_size+val_size:], y[train_size+val_size:]

print(f"\nData shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_val: {X_val.shape}")
print(f"  X_test: {X_test.shape}")

In [ ]:
# Modèle avec Attention Mechanism
def build_attention_model(input_shape, output_size):
    inputs = Input(shape=input_shape)
    
    # Bidirectional LSTM
    x = Bidirectional(LSTM(CONFIG["lstm_units"], return_sequences=True))(inputs)
    x = BatchNormalization()(x)
    x = Dropout(CONFIG["dropout_rate"])(x)
    
    # Self-Attention
    attention_output = MultiHeadAttention(
        num_heads=4, key_dim=32
    )(x, x)
    x = LayerNormalization()(x + attention_output)
    
    # Second LSTM
    x = LSTM(CONFIG["lstm_units"] // 2, return_sequences=False)(x)
    x = BatchNormalization()(x)
    x = Dropout(CONFIG["dropout_rate"])(x)
    
    # Dense layers
    x = Dense(64, activation='relu')(x)
    x = Dropout(CONFIG["dropout_rate"] / 2)(x)
    x = Dense(32, activation='relu')(x)
    
    # Output
    outputs = Dense(output_size)(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model

model = build_attention_model(
    input_shape=(X_train.shape[1], X_train.shape[2]),
    output_size=len(target_cols)
)

model.compile(
    optimizer=Adam(learning_rate=CONFIG["learning_rate"]),
    loss='huber',
    metrics=['mae']
)

model.summary()

In [ ]:
# Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_model_C.keras', monitor='val_loss', save_best_only=True, verbose=0)
]

print("Training Option C (External Data + Attention)...\n")

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=CONFIG['epochs'],
    batch_size=CONFIG['batch_size'],
    callbacks=callbacks,
    verbose=1
)

print("\nTraining complete!")

In [ ]:
# Visualiser l'entraînement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='Train')
axes[1].plot(history.history['val_mae'], label='Validation')
axes[1].set_title('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Évaluation
model = keras.models.load_model('best_model_C.keras')

y_pred_scaled = model.predict(X_test, verbose=0)
y_pred = scaler_targets.inverse_transform(y_pred_scaled)
y_true = scaler_targets.inverse_transform(y_test)

print("=" * 60)
print("OPTION C - RESULTATS (External Data + Attention)")
print("=" * 60)

metrics = {}
for i, col in enumerate(target_cols):
    mae = mean_absolute_error(y_true[:, i], y_pred[:, i])
    rmse = np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))
    mape = np.mean(np.abs((y_true[:, i] - y_pred[:, i]) / y_true[:, i])) * 100
    r2 = r2_score(y_true[:, i], y_pred[:, i])
    
    # Précision directionnelle
    true_dir = np.diff(y_true[:, i]) > 0
    pred_dir = np.diff(y_pred[:, i]) > 0
    dir_acc = np.mean(true_dir == pred_dir) * 100
    
    metrics[col] = {'mae': mae, 'rmse': rmse, 'mape': mape, 'r2': r2, 'directional_accuracy': dir_acc}
    
    print(f"\n{col}:")
    print(f"  MAE:  ${mae:,.2f}")
    print(f"  MAPE: {mape:.2f}%")
    print(f"  R2:   {r2:.4f}")
    print(f"  Direction Accuracy: {dir_acc:.1f}%")

In [ ]:
# Visualiser les prédictions
n_show = 60

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(target_cols):
    ax = axes[i]
    ax.plot(range(n_show), y_true[-n_show:, i], label='Actual', linewidth=2)
    ax.plot(range(n_show), y_pred[-n_show:, i], label='Predicted', linewidth=2, alpha=0.8)
    ax.set_title(f'{col} - Dir Acc: {metrics[col]["directional_accuracy"]:.1f}%')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Option C - External Data + Attention', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Analyser l'impact de Fear & Greed
if 'FearGreed' in df_features.columns:
    print("\n" + "="*60)
    print("ANALYSIS: Fear & Greed Impact")
    print("="*60)
    
    # Récupérer les Fear & Greed pour la période de test
    test_start = train_size + val_size + seq_len
    fng_test = df_features['FearGreed'].values[test_start:test_start+len(y_test)]
    
    # Analyser la performance par niveau de Fear & Greed
    # Extreme Fear: 0-25, Fear: 25-45, Neutral: 45-55, Greed: 55-75, Extreme Greed: 75-100
    
    close_idx = target_cols.index('Close')
    true_dir = np.diff(y_true[:, close_idx]) > 0
    pred_dir = np.diff(y_pred[:, close_idx]) > 0
    
    for level_name, (low, high) in [('Extreme Fear', (0, 25)), ('Fear', (25, 45)), 
                                      ('Neutral', (45, 55)), ('Greed', (55, 75)), 
                                      ('Extreme Greed', (75, 100))]:
        mask = (fng_test[:-1] >= low) & (fng_test[:-1] < high)
        if mask.sum() > 10:
            acc = np.mean(true_dir[mask] == pred_dir[mask]) * 100
            print(f"  {level_name} ({low}-{high}): {acc:.1f}% accuracy ({mask.sum()} samples)")

In [ ]:
# Sauvegarder
model.save('btc_lstm_model_C.keras')

scalers = {
    'scaler_features': scaler_features,
    'scaler_targets': scaler_targets
}
joblib.dump(scalers, 'btc_scaler_C.pkl')

config_save = {
    'model_version': '1.0.0-C-external',
    'trained_at': datetime.now().isoformat(),
    'option': 'C - External Data + Attention',
    'sequence_length': CONFIG['sequence_length'],
    'feature_columns': feature_cols,
    'target_columns': target_cols,
    'external_data_sources': ['Fear & Greed Index', 'S&P 500', 'Gold', 'VIX', 'Ethereum', 'Dollar Index'],
    'metrics': metrics,
}

with open('training_config_C.json', 'w') as f:
    json.dump(config_save, f, indent=2, default=str)

print("\nFiles saved:")
print("  - btc_lstm_model_C.keras")
print("  - btc_scaler_C.pkl")
print("  - training_config_C.json")

In [ ]:
# Télécharger
try:
    from google.colab import files
    files.download('btc_lstm_model_C.keras')
    files.download('btc_scaler_C.pkl')
    files.download('training_config_C.json')
except:
    print("Download manually from the file browser")

## Note Importante pour l'Option C

Pour utiliser ce modèle en production, le backend devra:

1. **Récupérer le Fear & Greed Index** via l'API Alternative.me
2. **Récupérer les prix des actifs corrélés** (S&P500, Gold, ETH, etc.)
3. **Calculer les mêmes features** que celles utilisées lors de l'entraînement

C'est plus complexe à mettre en production mais potentiellement plus précis.